# Version 2 NN Architecture Run and Review

Run the Torch feed-forward NN under `max_v3` only, one architecture at a time, so you can compare 1-layer, 2-layer, 3-layer, and 4-layer versions directly. Each architecture has 3 blocks: run, summary, and true-latest recommendations.

## Mount Drive and Set Root

Purpose: mount Google Drive in Colab, then point the notebook to the model project root.

In [9]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path
import subprocess, sys, json, copy
import pandas as pd
from IPython.display import display

root_candidates = [
    Path('/content/drive/MyDrive/v2/model'),
    Path('/content/v2/model'),
    Path.cwd().resolve().parents[0],
]
ROOT = next(path for path in root_candidates if path.exists())
BASE_CONFIG = ROOT / 'configs' / 'max_v3.yaml'
print({'ROOT': str(ROOT), 'BASE_CONFIG': str(BASE_CONFIG)})
ROOT

{'ROOT': '/content/drive/MyDrive/v2/model', 'BASE_CONFIG': '/content/drive/MyDrive/v2/model/configs/max_v3.yaml'}


PosixPath('/content/drive/MyDrive/v2/model')

## Install Requirements and Load Helpers

Purpose: install dependencies, load helper functions, and support running each NN architecture under the same `max_v3` feature set.

In [11]:
from pathlib import Path
import subprocess, sys, json
import pandas as pd
from IPython.display import display

root_candidates = [
    Path('/content/drive/MyDrive/v2/model'),
    Path('/content/v2/model'),
    Path.cwd().resolve().parents[0],
]
ROOT = next(path for path in root_candidates if path.exists())
AVAILABLE_PROFILES = ['careful_v3', 'max_v3']
DEFAULT_FEATURE_PROFILE = 'max_v3'
print({'ROOT': str(ROOT), 'available_profiles': AVAILABLE_PROFILES, 'default_profile': DEFAULT_FEATURE_PROFILE})
ROOT

{'ROOT': '/content/drive/MyDrive/v2/model', 'available_profiles': ['careful_v3', 'max_v3'], 'default_profile': 'max_v3'}


PosixPath('/content/drive/MyDrive/v2/model')

In [17]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import yaml
from v2_model.config import load_config
from v2_model.recommend_true_latest import build_true_latest_recommendations
from v2_model.recommend import build_latest_recommendations

ARCHITECTURES = {
    'NN_1L': [64],
    'NN_2L': [64, 32],
    'NN_3L': [64, 32, 16],
    'NN_4L': [64, 32, 16, 8],
}

LAST_RUN_DIRS = {}
LAST_CONFIG_PATHS = {}

def _parse_run_dir(stdout_text: str):
    marker = 'Pipeline completed. Outputs saved to:'
    for line in reversed(stdout_text.splitlines()):
        if marker in line:
            return Path(line.split(marker, 1)[1].strip())
    return None

def _manifest_architecture(run_dir: Path):
    manifest = run_dir / 'meta' / 'run_manifest.json'
    if not manifest.exists():
        return None
    try:
        payload = json.loads(manifest.read_text())
        nn_cfg = payload.get('config', {}).get('models', {}).get('nn', {})
        return tuple(nn_cfg.get('hidden_layer_grid', [[None]])[0])
    except Exception:
        return None

def _architecture_label(hidden_layers):
    hidden_layers = tuple(hidden_layers)
    for label, arch in ARCHITECTURES.items():
        if tuple(arch) == hidden_layers:
            return label
    raise KeyError(f'Unknown architecture: {hidden_layers}')

def _make_nn_config(label: str):
    hidden_layers = ARCHITECTURES[label]
    raw = yaml.safe_load(BASE_CONFIG.read_text())
    raw.setdefault('models', {})
    raw['models'].setdefault('nn', {})
    raw['models']['nn']['hidden_layer_grid'] = [hidden_layers]
    out_path = Path('/tmp') / f'{label.lower()}_max_v3.yaml'
    out_path.write_text(yaml.safe_dump(raw, sort_keys=False))
    LAST_CONFIG_PATHS[label] = out_path
    return out_path

def run_nn_architecture(label: str, stages: str = 'all'):
    config_path = _make_nn_config(label)
    cmd = [sys.executable, str(ROOT / 'run_model.py'), '--config', str(config_path), '--models', 'NN', '--stages', stages]
    print('Running:', ' '.join(map(str, cmd)))
    proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line.rstrip(''))
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'NN run failed with code {rc}')
    run_dir = _parse_run_dir(''.join(lines))
    if run_dir is None:
        raise RuntimeError('Could not parse run directory from command output.')
    LAST_RUN_DIRS[label] = run_dir
    return run_dir

def get_run_dir(label: str):
    if label in LAST_RUN_DIRS:
        return LAST_RUN_DIRS[label]
    candidates = sorted((ROOT / 'outputs').glob('run_*'))
    target_arch = tuple(ARCHITECTURES[label])
    for run_dir in reversed(candidates):
        if not (run_dir / 'r2' / 'nn_r2_summary_full_large_small.csv').exists():
            continue
        if _manifest_architecture(run_dir) != target_arch:
            continue
        LAST_RUN_DIRS[label] = run_dir
        return run_dir
    raise FileNotFoundError(f'No saved NN run found for {label}. Run the architecture first.')

def show_nn_summary(label: str):
    run_dir = get_run_dir(label)
    print('Architecture:', label, ARCHITECTURES[label])
    print('Run dir:', run_dir)
    display(pd.read_csv(run_dir / 'preprocess' / 'panel_prep_summary.csv'))
    display(pd.read_csv(run_dir / 'preprocess' / 'window_coverage_summary.csv'))
    display(pd.read_csv(run_dir / 'preprocess' / 'preprocess_report.csv'))
    display(pd.read_csv(run_dir / 'r2' / 'nn_r2_summary_full_large_small.csv'))
    display(pd.read_csv(run_dir / 'portfolio' / 'nn_performance_summary.csv'))
    display(pd.read_csv(run_dir / 'benchmark' / 'nn_vs_benchmark.csv'))
    imp_path = run_dir / 'importance' / 'nn_feature_importance.csv'
    if imp_path.exists():
        display(pd.read_csv(imp_path).head(15))
    comp_path = run_dir / 'complexity' / 'nn_complexity.csv'
    if comp_path.exists():
        display(pd.read_csv(comp_path).head(15))

def show_true_latest_nn_recommendations(label: str, top_k: int = 10, save_to_run_dir: bool = True):
    config_path = LAST_CONFIG_PATHS.get(label) or _make_nn_config(label)
    cfg = load_config(config_path)
    result = build_latest_recommendations(cfg, 'NN', top_k=top_k)
    print('Architecture:', label, ARCHITECTURES[label])
    print('Feature profile: max_v3')
    print('True latest month scored:', result.score_eom.date())
    print('Latest labeled month used for calibration:', result.latest_labeled_eom.date())
    print('Calibration window train:', result.train_start.date(), '->', result.train_end.date())
    print('Calibration window val  :', result.val_start.date(), '->', result.val_end.date())
    print('Eligible latest-month rows:', result.universe_rows)
    print('Scored latest-month rows  :', result.scored_rows)
    display(result.recommendations)
    if save_to_run_dir:
        run_dir = get_run_dir(label)
        out_path = run_dir / 'recommendations' / f"{label.lower()}_true_latest_recommendations.csv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        result.recommendations.to_csv(out_path, index=False)
        print('Saved:', out_path)



## 1 layer

This section runs the NN with `hidden_layer_grid = [[64]]` under `max_v3` only. Block 1 runs the model, Block 2 shows the summary, and Block 3 shows the true-latest recommended stocks.

In [14]:
RUN_DIR_NN_1L = run_nn_architecture('NN_1L')
RUN_DIR_NN_1L

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /tmp/nn_1l_max_v3.yaml --models NN --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffi

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T194950Z')

In [15]:
show_nn_summary('NN_1L')

Architecture: NN_1L [64]
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T194950Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_NN,n_rows,n_months
0,Full sample,-0.093118,12716,156
1,Large firms,-0.225623,3756,156
2,Small firms,-0.194321,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,NN_LS_EW_gross,2.334965,0,156,0.017880,0.067938,0.204076,0.235345,0.911709,10.181471,-0.276445,-0.224873
1,NN_LS_EW_net_0bps,2.334965,0,156,0.017880,0.067938,0.204076,0.235345,0.911709,10.181471,-0.276445,-0.224873
2,NN_LS_EW_net_10bps,2.334965,10,156,0.015546,0.067852,0.171283,0.235046,0.793660,6.809187,-0.306354,-0.226873
3,NN_LS_EW_net_20bps,2.334965,20,156,0.013211,0.067771,0.139307,0.234764,0.675260,4.449177,-0.339808,-0.228873
4,NN_LS_EW_net_30bps,2.334965,30,156,0.010876,0.067694,0.108129,0.234500,0.556534,2.799026,-0.371730,-0.230873
5,NN_LS_VW_gross,2.678050,0,156,0.012443,0.101846,0.087567,0.352805,0.423224,1.978029,-0.697286,-0.433110
6,NN_LS_VW_net_0bps,2.678050,0,156,0.012443,0.101846,0.087567,0.352805,0.423224,1.978029,-0.697286,-0.433110
7,NN_LS_VW_net_10bps,2.678050,10,156,0.009765,0.101940,0.053051,0.353129,0.331830,0.958122,-0.723153,-0.436570
8,NN_LS_VW_net_20bps,2.678050,20,156,0.007087,0.102042,0.019529,0.353482,0.240585,0.285860,-0.757408,-0.440031
9,NN_LS_VW_net_30bps,2.678050,30,156,0.004409,0.102152,-0.013024,0.353864,0.149509,-0.156690,-0.787806,-0.443492


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,NN_LS_EW_net_30bps,benchmark_equal_excess,156,0.003215,0.033568,-0.170265
1,value,NN_LS_VW_net_30bps,benchmark_value_excess,156,-0.003252,-0.028644,0.069879


,Feature,R2OOS,red_R2OOS,var_imp
0,dollar_vol,0.030807,0.013731,0.218823
1,be_me_x_roeq,0.031373,0.013165,0.209804
2,sales_growth_12m_raw,0.031388,0.013151,0.209572
3,intraday_range_vol_1m,0.031584,0.012954,0.206444
4,std_turn,0.032201,0.012338,0.196617
5,tri_ret_12_1,0.032603,0.011936,0.190212
6,equity_growth_12m,0.041426,0.003113,0.049605
7,ppe_growth_12m,0.041854,0.002684,0.042778
8,ebitda_assets_raw,0.042617,0.001921,0.030620
9,chcsho,0.043052,0.001486,0.023685


,window_id,test_start,test_end,best_score,best_hidden_layers,best_dropout,best_learning_rate,best_weight_decay,best_best_epoch,best_batch_size,best_epoch
0,1,2012-03-31,2013-02-28,0.136121,[64],0.1,0.001,0.00010,8,1024,8
1,2,2013-03-31,2014-02-28,0.136299,[64],0.1,0.001,0.00010,6,1024,6
2,3,2014-03-31,2015-02-28,0.131559,[64],0.1,0.001,0.00010,6,1024,6
3,4,2015-03-31,2016-02-29,0.119077,[64],0.1,0.001,0.00010,15,1024,15
4,5,2016-03-31,2017-02-28,0.117328,[64],0.1,0.001,0.00010,12,1024,12
5,6,2017-03-31,2018-02-28,0.141185,[64],0.1,0.001,0.00010,21,1024,21
6,7,2018-03-31,2019-02-28,0.148664,[64],0.1,0.001,0.00010,23,1024,23
7,8,2019-03-31,2020-02-29,0.136048,[64],0.1,0.001,0.00010,22,1024,22
8,9,2020-03-31,2021-02-28,0.129170,[64],0.1,0.001,0.00010,8,1024,8
9,10,2021-03-31,2022-02-28,0.156508,[64],0.1,0.001,0.00010,10,1024,10


## 2 layers

This section runs the NN with `hidden_layer_grid = [[64, 32]]` under `max_v3` only. Block 1 runs the model, Block 2 shows the summary, and Block 3 shows the true-latest recommended stocks.

In [20]:
RUN_DIR_NN_2L = run_nn_architecture('NN_2L')
RUN_DIR_NN_2L

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /tmp/nn_2l_max_v3.yaml --models NN --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffi

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T200935Z')

In [21]:
show_nn_summary('NN_2L')

Architecture: NN_2L [64, 32]
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T200935Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_NN,n_rows,n_months
0,Full sample,-0.013184,12716,156
1,Large firms,-0.052059,3756,156
2,Small firms,-0.024956,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,NN_LS_EW_gross,2.238552,0,156,0.036151,0.067596,0.492856,0.234159,1.852637,181.908673,-0.326605,-0.173009
1,NN_LS_EW_net_0bps,2.238552,0,156,0.036151,0.067596,0.492856,0.234159,1.852637,181.908673,-0.326605,-0.173009
2,NN_LS_EW_net_10bps,2.238552,10,156,0.033912,0.067501,0.454562,0.233830,1.740360,129.472602,-0.347749,-0.174646
3,NN_LS_EW_net_20bps,2.238552,20,156,0.031674,0.067413,0.417163,0.233524,1.627613,91.994705,-0.368280,-0.176282
4,NN_LS_EW_net_30bps,2.238552,30,156,0.029435,0.067330,0.380641,0.233239,1.514426,65.229158,-0.388213,-0.177918
5,NN_LS_VW_gross,2.765440,0,156,0.035274,0.167459,0.355970,0.580093,0.729684,51.390513,-0.666101,-0.372167
6,NN_LS_VW_net_0bps,2.765440,0,156,0.035274,0.167459,0.355970,0.580093,0.729684,51.390513,-0.666101,-0.372167
7,NN_LS_VW_net_10bps,2.765440,10,156,0.032508,0.167496,0.312324,0.580224,0.672326,33.240125,-0.675737,-0.375074
8,NN_LS_VW_net_20bps,2.765440,20,156,0.029743,0.167538,0.269956,0.580370,0.614978,21.348832,-0.685125,-0.377981
9,NN_LS_VW_net_30bps,2.765440,30,156,0.026977,0.167585,0.228833,0.580530,0.557644,13.568250,-0.694272,-0.380888


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,NN_LS_EW_net_30bps,benchmark_equal_excess,156,0.021775,0.246093,-0.003185
1,value,NN_LS_VW_net_30bps,benchmark_value_excess,156,0.019317,0.110948,0.054417


,Feature,R2OOS,red_R2OOS,var_imp
0,cash_ratio,0.034933,-0.005487,0.027792
1,ep,0.034841,-0.005395,0.027330
2,opercf_assets_raw,0.034460,-0.005014,0.025400
3,ni_assets_raw,0.034454,-0.005008,0.025367
4,gma,0.034297,-0.004851,0.024571
5,cur_liab_growth_12m,0.034287,-0.004841,0.024520
6,Free_Float_Pct,0.034284,-0.004838,0.024506
7,cash_assets_raw,0.034105,-0.004659,0.023599
8,std_turn,0.034070,-0.004624,0.023422
9,mom36m_x_idiovol,0.034033,-0.004587,0.023234


,window_id,test_start,test_end,best_score,best_hidden_layers,best_dropout,best_learning_rate,best_weight_decay,best_best_epoch,best_batch_size,best_epoch
0,1,2012-03-31,2013-02-28,0.127414,"[64, 32]",0.1,0.001,0.00010,6,1024,6
1,2,2013-03-31,2014-02-28,0.126570,"[64, 32]",0.1,0.001,0.00010,8,1024,8
2,3,2014-03-31,2015-02-28,0.125797,"[64, 32]",0.1,0.001,0.00001,14,1024,14
3,4,2015-03-31,2016-02-29,0.113049,"[64, 32]",0.1,0.001,0.00010,10,1024,10
4,5,2016-03-31,2017-02-28,0.108887,"[64, 32]",0.1,0.001,0.00010,3,1024,3
5,6,2017-03-31,2018-02-28,0.133488,"[64, 32]",0.1,0.001,0.00010,3,1024,3
6,7,2018-03-31,2019-02-28,0.144015,"[64, 32]",0.1,0.001,0.00010,6,1024,6
7,8,2019-03-31,2020-02-29,0.131451,"[64, 32]",0.1,0.001,0.00010,4,1024,4
8,9,2020-03-31,2021-02-28,0.127004,"[64, 32]",0.1,0.001,0.00001,6,1024,6
9,10,2021-03-31,2022-02-28,0.152139,"[64, 32]",0.1,0.001,0.00010,5,1024,5


## 3 layers

This section runs the NN with `hidden_layer_grid = [[64, 32, 16]]` under `max_v3` only. Block 1 runs the model, Block 2 shows the summary, and Block 3 shows the true-latest recommended stocks.

In [22]:
RUN_DIR_NN_3L = run_nn_architecture('NN_3L')
RUN_DIR_NN_3L

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /tmp/nn_3l_max_v3.yaml --models NN --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffi

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T202026Z')

In [23]:
show_nn_summary('NN_3L')

Architecture: NN_3L [64, 32, 16]
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T202026Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_NN,n_rows,n_months
0,Full sample,-0.039257,12716,156
1,Large firms,-0.072526,3756,156
2,Small firms,-0.054942,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,NN_LS_EW_gross,1.989876,0,156,0.026175,0.070600,0.325464,0.244567,1.284285,37.974688,-0.359892,-0.192524
1,NN_LS_EW_net_0bps,1.989876,0,156,0.026175,0.070600,0.325464,0.244567,1.284285,37.974688,-0.359892,-0.192524
2,NN_LS_EW_net_10bps,1.989876,10,156,0.024185,0.070571,0.294845,0.244464,1.187153,27.762660,-0.376601,-0.194342
3,NN_LS_EW_net_20bps,1.989876,20,156,0.022195,0.070549,0.264866,0.244388,1.089814,20.211938,-0.392910,-0.196161
4,NN_LS_EW_net_30bps,1.989876,30,156,0.020205,0.070535,0.235517,0.244339,0.992303,14.632737,-0.409056,-0.197979
5,NN_LS_VW_gross,2.421813,0,156,0.030279,0.097363,0.355600,0.337275,1.077304,51.204891,-0.398580,-0.223082
6,NN_LS_VW_net_0bps,2.421813,0,156,0.030279,0.097363,0.355600,0.337275,1.077304,51.204891,-0.398580,-0.223082
7,NN_LS_VW_net_10bps,2.421813,10,156,0.027857,0.097368,0.317480,0.337294,0.991084,35.030769,-0.417001,-0.224887
8,NN_LS_VW_net_20bps,2.421813,20,156,0.025435,0.097382,0.280331,0.337342,0.904793,23.842212,-0.444900,-0.226691
9,NN_LS_VW_net_30bps,2.421813,30,156,0.023014,0.097405,0.244131,0.337420,0.818453,16.110341,-0.472558,-0.228496


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,NN_LS_EW_net_30bps,benchmark_equal_excess,156,0.012544,0.140486,0.033875
1,value,NN_LS_VW_net_30bps,benchmark_value_excess,156,0.015353,0.138416,0.040933


,Feature,R2OOS,red_R2OOS,var_imp
0,be_me,0.015180,0.000980,0.117192
1,tri_mom36m,0.015275,0.000885,0.105881
2,oc_ret_1m,0.015304,0.000856,0.102417
3,gma,0.015374,0.000786,0.094029
4,ret_12_1_x_vn_mkt_chg1m,0.015416,0.000744,0.088940
5,be_me_x_roeq,0.015433,0.000727,0.086973
6,cash_growth_12m,0.015513,0.000647,0.077433
7,cash_ratio,0.015572,0.000588,0.070309
8,grossprofit_assets_raw,0.015579,0.000581,0.069438
9,Vol_30D,0.015594,0.000566,0.067668


,window_id,test_start,test_end,best_score,best_hidden_layers,best_dropout,best_learning_rate,best_weight_decay,best_best_epoch,best_batch_size,best_epoch
0,1,2012-03-31,2013-02-28,0.126452,"[64, 32, 16]",0.0,0.001,0.00010,23,1024,23
1,2,2013-03-31,2014-02-28,0.126236,"[64, 32, 16]",0.1,0.001,0.00010,17,1024,17
2,3,2014-03-31,2015-02-28,0.124492,"[64, 32, 16]",0.1,0.001,0.00010,4,1024,4
3,4,2015-03-31,2016-02-29,0.111763,"[64, 32, 16]",0.1,0.001,0.00001,11,1024,11
4,5,2016-03-31,2017-02-28,0.107624,"[64, 32, 16]",0.1,0.001,0.00001,7,1024,7
5,6,2017-03-31,2018-02-28,0.132916,"[64, 32, 16]",0.1,0.001,0.00001,10,1024,10
6,7,2018-03-31,2019-02-28,0.143086,"[64, 32, 16]",0.1,0.001,0.00010,6,1024,6
7,8,2019-03-31,2020-02-29,0.129667,"[64, 32, 16]",0.0,0.001,0.00001,4,1024,4
8,9,2020-03-31,2021-02-28,0.124327,"[64, 32, 16]",0.0,0.001,0.00001,4,1024,4
9,10,2021-03-31,2022-02-28,0.152573,"[64, 32, 16]",0.1,0.001,0.00010,17,1024,17


## 4 layers

This section runs the NN with `hidden_layer_grid = [[64, 32, 16, 8]]` under `max_v3` only. Block 1 runs the model, Block 2 shows the summary, and Block 3 shows the true-latest recommended stocks.

In [24]:
RUN_DIR_NN_4L = run_nn_architecture('NN_4L')
RUN_DIR_NN_4L

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /tmp/nn_4l_max_v3.yaml --models NN --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffi

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T203218Z')

In [25]:
show_nn_summary('NN_4L')

Architecture: NN_4L [64, 32, 16, 8]
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T203218Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_NN,n_rows,n_months
0,Full sample,-0.039818,12716,156
1,Large firms,-0.088685,3756,156
2,Small firms,-0.036482,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,NN_LS_EW_gross,1.778618,0,156,0.026909,0.076547,0.332390,0.265167,1.217737,40.706836,-0.350641,-0.120335
1,NN_LS_EW_net_0bps,1.778618,0,156,0.026909,0.076547,0.332390,0.265167,1.217737,40.706836,-0.350641,-0.120335
2,NN_LS_EW_net_10bps,1.778618,10,156,0.025130,0.076478,0.304892,0.264926,1.138281,30.802997,-0.359697,-0.122501
3,NN_LS_EW_net_20bps,1.778618,20,156,0.023351,0.076415,0.277907,0.264708,1.058588,23.237620,-0.368648,-0.124668
4,NN_LS_EW_net_30bps,1.778618,30,156,0.021573,0.076358,0.251427,0.264513,0.978679,17.461709,-0.377495,-0.126835
5,NN_LS_VW_gross,2.151401,0,156,0.023892,0.149593,0.203790,0.518207,0.553269,10.146995,-0.477845,-0.354055
6,NN_LS_VW_net_0bps,2.151401,0,156,0.023892,0.149593,0.203790,0.518207,0.553269,10.146995,-0.477845,-0.354055
7,NN_LS_VW_net_10bps,2.151401,10,156,0.021741,0.149555,0.173350,0.518072,0.503581,6.990185,-0.493051,-0.356479
8,NN_LS_VW_net_20bps,2.151401,20,156,0.019590,0.149523,0.143602,0.517962,0.453845,4.722319,-0.522511,-0.358903
9,NN_LS_VW_net_30bps,2.151401,30,156,0.017438,0.149498,0.114532,0.517876,0.404069,3.094503,-0.550340,-0.361326


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,NN_LS_EW_net_30bps,benchmark_equal_excess,156,0.013912,0.151476,0.076306
1,value,NN_LS_VW_net_30bps,benchmark_value_excess,156,0.009777,0.061399,0.015372


,Feature,R2OOS,red_R2OOS,var_imp
0,ev_ebitda_raw,0.018187,-0.001360,0.043509
1,std_turn,0.018004,-0.001177,0.037658
2,idiovol,0.017787,-0.000960,0.030729
3,cfp_x_lev,0.017771,-0.000944,0.030197
4,opercf_assets_raw,0.017763,-0.000936,0.029959
5,mom1m_x_idiovol,0.017740,-0.000913,0.029221
6,ebitda_growth_12m_raw,0.017722,-0.000895,0.028646
7,grossprofit_assets_raw,0.017551,-0.000724,0.023161
8,intraday_range_vol_1m,0.017546,-0.000719,0.022996
9,me,0.017531,-0.000704,0.022517


,window_id,test_start,test_end,best_score,best_hidden_layers,best_dropout,best_learning_rate,best_weight_decay,best_best_epoch,best_batch_size,best_epoch
0,1,2012-03-31,2013-02-28,0.125496,"[64, 32, 16, 8]",0.0,0.001,0.00001,27,1024,27
1,2,2013-03-31,2014-02-28,0.125611,"[64, 32, 16, 8]",0.1,0.001,0.00010,22,1024,22
2,3,2014-03-31,2015-02-28,0.121660,"[64, 32, 16, 8]",0.1,0.001,0.00010,6,1024,6
3,4,2015-03-31,2016-02-29,0.111179,"[64, 32, 16, 8]",0.0,0.001,0.00001,4,1024,4
4,5,2016-03-31,2017-02-28,0.107702,"[64, 32, 16, 8]",0.0,0.001,0.00010,11,1024,11
5,6,2017-03-31,2018-02-28,0.132927,"[64, 32, 16, 8]",0.1,0.001,0.00001,13,1024,13
6,7,2018-03-31,2019-02-28,0.143012,"[64, 32, 16, 8]",0.1,0.001,0.00001,17,1024,17
7,8,2019-03-31,2020-02-29,0.129548,"[64, 32, 16, 8]",0.0,0.001,0.00010,6,1024,6
8,9,2020-03-31,2021-02-28,0.123972,"[64, 32, 16, 8]",0.0,0.001,0.00001,5,1024,5
9,10,2021-03-31,2022-02-28,0.152109,"[64, 32, 16, 8]",0.1,0.001,0.00010,25,1024,25
